In [6]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not ((ROOT / "data").is_dir() and (ROOT / "src").is_dir()):
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Project root not found.")
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.feature_analysis import load_data, align_test_sensors
from src.lightgbm_model import get_base_features
from src.cv_utils import load_folds

# Load data and align test sensors exactly once.
train_df, test_df = load_data()
train_df = train_df.reset_index(drop=True)
test_aligned = align_test_sensors(test_df, train_df)

features = get_base_features(
    train_df,
    test_aligned,
    test_already_aligned=True,
)

# Repair-cost modelling population.
failed = train_df[
    (train_df["failure"] == 1) &
    (train_df["repair_cost"].notna())
].copy()

folds = load_folds()

print("=" * 55)
print("REPAIR-COST DATA AUDIT")
print("=" * 55)

print(f"Training vehicles: {len(train_df):,}")
print(f"Test vehicles: {len(test_df):,}")
print(f"Failed vehicles: {(train_df['failure'] == 1).sum():,}")
print(f"Known repair costs: {len(failed):,}")
print(f"Available sensor features: {len(features)}")

print("\nREPAIR COST DISTRIBUTION")
display(
    failed["repair_cost"].describe(
        percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    ).to_frame()
)

print("\nREPAIR COST QUARTILES")
failed["cost_quartile"] = pd.qcut(
    failed["repair_cost"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"],
    duplicates="drop",
)
display(
    failed.groupby("cost_quartile", observed=True)["repair_cost"]
    .agg(["count", "min", "median", "mean", "max"])
)

print("\nKNOWN REPAIR COSTS PER EXISTING CV FOLD")
fold_rows = []

for fold_no, (train_idx, valid_idx) in enumerate(folds):
    fold_failed = train_df.iloc[valid_idx]
    fold_failed = fold_failed[
        (fold_failed["failure"] == 1) &
        fold_failed["repair_cost"].notna()
    ]

    fold_rows.append({
        "fold": fold_no,
        "known_cost_cases": len(fold_failed),
        "mean_cost": fold_failed["repair_cost"].mean(),
        "median_cost": fold_failed["repair_cost"].median(),
        "max_cost": fold_failed["repair_cost"].max(),
    })

display(pd.DataFrame(fold_rows).round(2))

print("\nMISSINGNESS AMONG FAILED VEHICLES")
missingness = (
    failed[features].isna().mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .to_frame()
)
display(missingness.head(20))

REPAIR-COST DATA AUDIT
Training vehicles: 56,000
Test vehicles: 20,000
Failed vehicles: 397
Known repair costs: 397
Available sensor features: 155

REPAIR COST DISTRIBUTION


,repair_cost
count,397.000000
mean,492.382872
std,993.732983
min,5.000000
10%,16.600000
25%,71.000000
50%,176.000000
75%,488.000000
90%,1094.400000
95%,1693.200000



REPAIR COST QUARTILES


,count,min,median,mean,max
cost_quartile,,,,,
Q1,101,5.0,25.0,28.683168,71.0
Q2,99,74.0,117.0,121.676768,176.0
Q3,98,177.0,305.0,310.948980,488.0
Q4,99,492.0,991.0,1515.757576,8000.0



KNOWN REPAIR COSTS PER EXISTING CV FOLD


,fold,known_cost_cases,mean_cost,median_cost,max_cost
0,0,79,341.08,156.0,2051.0
1,1,79,398.73,196.0,4923.0
2,2,79,575.27,169.0,8000.0
3,3,80,670.66,184.0,8000.0
4,4,80,474.15,218.0,6842.0



MISSINGNESS AMONG FAILED VEHICLES


,missing_fraction
s021,0.773300
s147,0.773300
s002,0.695214
s150,0.695214
s125,0.695214
s036,0.695214
s169,0.695214
s016,0.589421
s076,0.589421
s093,0.589421


In [7]:
from scipy.stats import spearmanr
from IPython.display import display

# Use only failed vehicles with known repair costs.
X = failed[features].copy()
y = failed["repair_cost"].astype(float)
log_y = np.log1p(y)

rows = []

for col in features:
    values = X[col]
    observed = values.notna()
    missing = values.isna()

    # Require enough observations for a meaningful association estimate.
    if observed.sum() < 30:
        continue

    # Association between observed sensor readings and repair cost.
    raw_corr = spearmanr(
        values[observed],
        y[observed]
    ).statistic

    log_corr = spearmanr(
        values[observed],
        log_y[observed]
    ).statistic

    # Does missingness itself carry repair-cost information?
    missing_corr = (
        spearmanr(missing.astype(int), y).statistic
        if missing.nunique() > 1 else np.nan
    )

    present_median = y[observed].median()
    missing_median = y[missing].median() if missing.any() else np.nan

    rows.append({
        "sensor": col,
        "observed_count": int(observed.sum()),
        "missing_pct": missing.mean() * 100,
        "reading_cost_corr": raw_corr,
        "log_cost_corr": log_corr,
        "missingness_corr": missing_corr,
        "present_cost_median": present_median,
        "missing_cost_median": missing_median,
        "median_difference": missing_median - present_median
            if missing.any() else np.nan,
    })

association = pd.DataFrame(rows).set_index("sensor")

print("TOP SENSOR READINGS ASSOCIATED WITH REPAIR COST")
display(
    association.reindex(
        association["log_cost_corr"].abs()
        .sort_values(ascending=False).index
    ).head(20).round(3)
)

print("\nTOP MISSINGNESS ASSOCIATIONS")
display(
    association.reindex(
        association["missingness_corr"].abs()
        .sort_values(ascending=False).index
    ).head(20).round(3)
)

print("\nTOP SENSORS BY DIFFERENCE IN MEDIAN REPAIR COST")
display(
    association.reindex(
        association["median_difference"].abs()
        .sort_values(ascending=False).index
    ).head(20).round(3)
)

TOP SENSOR READINGS ASSOCIATED WITH REPAIR COST


C:\Users\Avnish\AppData\Local\Temp\ipykernel_22888\3673299096.py:21: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  raw_corr = spearmanr(
C:\Users\Avnish\AppData\Local\Temp\ipykernel_22888\3673299096.py:26: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  log_corr = spearmanr(
C:\Users\Avnish\AppData\Local\Temp\ipykernel_22888\3673299096.py:21: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  raw_corr = spearmanr(
C:\Users\Avnish\AppData\Local\Temp\ipykernel_22888\3673299096.py:26: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  log_corr = spearmanr(
C:\Users\Avnish\AppData\Local\Temp\ipykernel_22888\3673299096.py:21: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  raw_corr = spearmanr(
C:\Users\Avnish\AppData\Local\Temp\ipykernel_22888\3673299096.py:

,observed_count,missing_pct,reading_cost_corr,log_cost_corr,missingness_corr,present_cost_median,missing_cost_median,median_difference
sensor,,,,,,,,
s084,394,0.756,-0.458,-0.458,0.047,176.0,369.0,193.0
s083,394,0.756,-0.448,-0.448,0.047,176.0,369.0,193.0
s082,394,0.756,-0.442,-0.442,0.047,176.0,369.0,193.0
s081,394,0.756,-0.438,-0.438,0.047,176.0,369.0,193.0
s080,394,0.756,-0.425,-0.425,0.047,176.0,369.0,193.0
s058,395,0.504,-0.405,-0.405,0.066,176.0,650.5,474.5
s022,183,53.904,-0.361,-0.361,-0.062,202.0,155.5,-46.5
s142,183,53.904,-0.360,-0.360,-0.062,202.0,155.5,-46.5
s037,394,0.756,-0.357,-0.357,0.047,176.0,369.0,193.0



TOP MISSINGNESS ASSOCIATIONS


,observed_count,missing_pct,reading_cost_corr,log_cost_corr,missingness_corr,present_cost_median,missing_cost_median,median_difference
sensor,,,,,,,,
s017,163,58.942,-0.186,-0.186,-0.248,271.0,135.0,-136.0
s016,163,58.942,-0.060,-0.060,-0.248,271.0,135.0,-136.0
s095,163,58.942,0.018,0.018,-0.248,271.0,135.0,-136.0
s165,163,58.942,-0.069,-0.069,-0.248,271.0,135.0,-136.0
s119,163,58.942,0.151,0.151,-0.248,271.0,135.0,-136.0
s070,163,58.942,0.022,0.022,-0.248,271.0,135.0,-136.0
s093,163,58.942,-0.063,-0.063,-0.248,271.0,135.0,-136.0
s076,163,58.942,-0.001,-0.001,-0.248,271.0,135.0,-136.0
s090,163,58.942,-0.052,-0.052,-0.248,271.0,135.0,-136.0



TOP SENSORS BY DIFFERENCE IN MEDIAN REPAIR COST


,observed_count,missing_pct,reading_cost_corr,log_cost_corr,missingness_corr,present_cost_median,missing_cost_median,median_difference
sensor,,,,,,,,
s023,395,0.504,0.144,0.144,0.066,176.0,650.5,474.5
s025,395,0.504,0.110,0.110,0.066,176.0,650.5,474.5
s024,395,0.504,0.124,0.124,0.066,176.0,650.5,474.5
s156,395,0.504,-0.061,-0.061,0.066,176.0,650.5,474.5
s157,395,0.504,-0.108,-0.108,0.066,176.0,650.5,474.5
s158,395,0.504,-0.160,-0.160,0.066,176.0,650.5,474.5
s160,395,0.504,-0.160,-0.160,0.066,176.0,650.5,474.5
s159,395,0.504,-0.188,-0.188,0.066,176.0,650.5,474.5
s161,395,0.504,0.008,0.008,0.066,176.0,650.5,474.5


In [8]:
# 1. Correlation between sensor readings.
# Require sufficient overlapping observations.
sensor_corr = X.corr(method="spearman", min_periods=50)

upper = sensor_corr.where(
    np.triu(np.ones(sensor_corr.shape), k=1).astype(bool)
)

pairs = (
    upper.stack()
    .rename("correlation")
    .reset_index()
    .rename(columns={"level_0": "sensor_1", "level_1": "sensor_2"})
)

pairs["abs_correlation"] = pairs["correlation"].abs()

print("MOST REDUNDANT SENSOR PAIRS")
display(
    pairs.sort_values("abs_correlation", ascending=False)
    .head(25)
    .round(3)
)

# 2. Correlation between missingness indicators.
missing_corr = X.isna().astype(int).corr()

upper_missing = missing_corr.where(
    np.triu(np.ones(missing_corr.shape), k=1).astype(bool)
)

missing_pairs = (
    upper_missing.stack()
    .rename("missingness_correlation")
    .reset_index()
    .rename(columns={"level_0": "sensor_1", "level_1": "sensor_2"})
)

print("\nMOST SIMILAR MISSINGNESS PATTERNS")
display(
    missing_pairs.sort_values(
        "missingness_correlation",
        ascending=False
    ).head(25).round(3)
)

# 3. Missing sample counts for the strongest associations.
print("\nMISSINGNESS COUNTS FOR TOP SENSOR ASSOCIATIONS")
top_sensors = association["log_cost_corr"].abs().nlargest(20).index

missing_counts = pd.DataFrame({
    "missing": X[top_sensors].isna().sum(),
    "present": X[top_sensors].notna().sum(),
    "missing_cost_median": [
        y[X[col].isna()].median() for col in top_sensors
    ],
    "present_cost_median": [
        y[X[col].notna()].median() for col in top_sensors
    ],
})

display(missing_counts)

MOST REDUNDANT SENSOR PAIRS


,sensor_1,sensor_2,correlation,abs_correlation
17786,s126,s128,1.000,1.000
10981,s079,s144,1.000,1.000
1974,s012,s126,1.000,1.000
1976,s012,s128,1.000,1.000
5177,s034,s071,1.000,1.000
3097,s019,s167,1.000,1.000
248,s001,s102,1.000,1.000
633,s004,s013,1.000,1.000
1491,s009,s105,0.996,0.996
1332,s008,s101,0.995,0.995



MOST SIMILAR MISSINGNESS PATTERNS


,sensor_1,sensor_2,missingness_correlation
3616,s023,s054,1.0
3617,s023,s058,1.0
3618,s023,s060,1.0
3619,s023,s061,1.0
22154,s157,s159,1.0
22155,s157,s160,1.0
22156,s157,s161,1.0
22157,s157,s162,1.0
22158,s157,s163,1.0
22309,s158,s159,1.0



MISSINGNESS COUNTS FOR TOP SENSOR ASSOCIATIONS


,missing,present,missing_cost_median,present_cost_median
s084,3,394,369.0,176.0
s083,3,394,369.0,176.0
s082,3,394,369.0,176.0
s081,3,394,369.0,176.0
s080,3,394,369.0,176.0
s058,2,395,650.5,176.0
s022,214,183,155.5,202.0
s142,214,183,155.5,202.0
s037,3,394,369.0,176.0
s085,3,394,369.0,176.0


In [9]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Use the same existing five folds.
model_results = []

known_mask = (
    train_df["failure"].eq(1)
    & train_df["repair_cost"].notna()
)

for model_type in ["raw", "raw_plus_missing"]:
    oof = np.full(len(train_df), np.nan)

    for fold_no, (train_idx, valid_idx) in enumerate(folds, 1):

        train_idx = train_idx[known_mask.iloc[train_idx].to_numpy()]
        valid_idx = valid_idx[known_mask.iloc[valid_idx].to_numpy()]

        X_train = train_df.iloc[train_idx][features].copy()
        X_valid = train_df.iloc[valid_idx][features].copy()

        y_train = train_df.iloc[train_idx]["repair_cost"]

        if model_type == "raw_plus_missing":
            X_train = pd.concat(
                [X_train, X_train.isna().astype("int8").add_suffix("_missing")],
                axis=1,
            )
            X_valid = pd.concat(
                [X_valid, X_valid.isna().astype("int8").add_suffix("_missing")],
                axis=1,
            )

        model = LGBMRegressor(
            objective="regression",
            n_estimators=200,
            learning_rate=0.03,
            num_leaves=7,
            min_child_samples=15,
            reg_lambda=5.0,
            verbosity=-1,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        oof[valid_idx] = np.maximum(model.predict(X_valid), 0)

        print(f"{model_type}: fold {fold_no}/5 complete")

    valid = known_mask.to_numpy()
    actual = train_df.loc[valid, "repair_cost"].to_numpy()
    predicted = oof[valid]

    under = np.maximum(actual - predicted, 0)
    over = np.maximum(predicted - actual, 0)

    # Asymmetric budget loss, excluding fixed inspection cost.
    budget_loss = 0.20 * under + 0.02 * over

    expensive = actual >= np.quantile(actual, 0.75)

    model_results.append({
        "model": model_type,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "underestimation_rate": np.mean(predicted < actual),
        "asymmetric_loss": budget_loss.mean(),
        "expensive_quartile_MAE": mean_absolute_error(
            actual[expensive], predicted[expensive]
        ),
        "mean_prediction": predicted.mean(),
        "mean_actual": actual.mean(),
    })

display(pd.DataFrame(model_results).set_index("model").round(3))

raw: fold 1/5 complete
raw: fold 2/5 complete
raw: fold 3/5 complete
raw: fold 4/5 complete
raw: fold 5/5 complete
raw_plus_missing: fold 1/5 complete
raw_plus_missing: fold 2/5 complete
raw_plus_missing: fold 3/5 complete
raw_plus_missing: fold 4/5 complete
raw_plus_missing: fold 5/5 complete


,MAE,RMSE,underestimation_rate,asymmetric_loss,expensive_quartile_MAE,mean_prediction,mean_actual
model,,,,,,,
raw,401.691,866.361,0.39,45.721,868.124,475.326,492.383
raw_plus_missing,401.691,866.361,0.39,45.721,868.124,475.326,492.383


In [10]:
log_oof = np.full(len(train_df), np.nan)

for fold_no, (train_idx, valid_idx) in enumerate(folds, 1):

    train_idx = train_idx[
        known_mask.iloc[train_idx].to_numpy()
    ]
    valid_idx = valid_idx[
        known_mask.iloc[valid_idx].to_numpy()
    ]

    X_train = train_df.iloc[train_idx][features]
    X_valid = train_df.iloc[valid_idx][features]

    y_train = np.log1p(
        train_df.iloc[train_idx]["repair_cost"]
    )

    model = LGBMRegressor(
        objective="regression",
        n_estimators=200,
        learning_rate=0.03,
        num_leaves=7,
        min_child_samples=15,
        reg_lambda=5.0,
        verbosity=-1,
        random_state=42,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    log_oof[valid_idx] = np.maximum(
        np.expm1(model.predict(X_valid)), 0
    )

    print(f"Log-target: fold {fold_no}/5 complete")

valid = known_mask.to_numpy()
actual = train_df.loc[valid, "repair_cost"].to_numpy()
predicted = log_oof[valid]

under = np.maximum(actual - predicted, 0)
over = np.maximum(predicted - actual, 0)

expensive = actual >= np.quantile(actual, 0.75)

log_result = pd.DataFrame([{
    "model": "log_target",
    "MAE": mean_absolute_error(actual, predicted),
    "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
    "underestimation_rate": np.mean(predicted < actual),
    "asymmetric_loss": np.mean(0.20 * under + 0.02 * over),
    "expensive_quartile_MAE": mean_absolute_error(
        actual[expensive], predicted[expensive]
    ),
    "mean_prediction": predicted.mean(),
    "mean_actual": actual.mean(),
}])

display(log_result.round(3))

Log-target: fold 1/5 complete
Log-target: fold 2/5 complete
Log-target: fold 3/5 complete
Log-target: fold 4/5 complete
Log-target: fold 5/5 complete


,model,MAE,RMSE,underestimation_rate,asymmetric_loss,expensive_quartile_MAE,mean_prediction,mean_actual
0,log_target,329.375,887.246,0.509,55.5,990.182,278.281,492.383


In [11]:
quantile_results = []

actual = train_df.loc[known_mask, "repair_cost"].to_numpy()
valid_positions = np.flatnonzero(known_mask.to_numpy())

for alpha in [0.85, 0.90]:

    quantile_oof = np.full(len(train_df), np.nan)

    for fold_no, (train_idx, valid_idx) in enumerate(folds, 1):

        train_idx = train_idx[
            known_mask.iloc[train_idx].to_numpy()
        ]
        valid_idx = valid_idx[
            known_mask.iloc[valid_idx].to_numpy()
        ]

        X_train = train_df.iloc[train_idx][features]
        X_valid = train_df.iloc[valid_idx][features]
        y_train = train_df.iloc[train_idx]["repair_cost"]

        model = LGBMRegressor(
            objective="quantile",
            alpha=alpha,
            n_estimators=200,
            learning_rate=0.03,
            num_leaves=7,
            min_child_samples=15,
            reg_lambda=5.0,
            verbosity=-1,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        quantile_oof[valid_idx] = np.maximum(
            model.predict(X_valid), 0
        )

        print(f"Alpha {alpha}: fold {fold_no}/5 complete")

    predicted = quantile_oof[valid_positions]

    under = np.maximum(actual - predicted, 0)
    over = np.maximum(predicted - actual, 0)
    expensive = actual >= np.quantile(actual, 0.75)

    quantile_results.append({
        "alpha": alpha,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "underestimation_rate": np.mean(predicted < actual),
        "asymmetric_loss": np.mean(0.20 * under + 0.02 * over),
        "expensive_quartile_MAE": mean_absolute_error(
            actual[expensive], predicted[expensive]
        ),
        "mean_prediction": predicted.mean(),
    })

display(pd.DataFrame(quantile_results).set_index("alpha").round(3))

Alpha 0.85: fold 1/5 complete
Alpha 0.85: fold 2/5 complete
Alpha 0.85: fold 3/5 complete
Alpha 0.85: fold 4/5 complete
Alpha 0.85: fold 5/5 complete
Alpha 0.9: fold 1/5 complete
Alpha 0.9: fold 2/5 complete
Alpha 0.9: fold 3/5 complete
Alpha 0.9: fold 4/5 complete
Alpha 0.9: fold 5/5 complete


,MAE,RMSE,underestimation_rate,asymmetric_loss,expensive_quartile_MAE,mean_prediction
alpha,,,,,,
0.85,548.846,931.784,0.179,40.424,946.231,714.041
0.90,688.792,1083.156,0.149,39.445,982.394,895.963


In [12]:
from catboost import CatBoostRegressor

cat_oof = np.full(len(train_df), np.nan)

for fold_no, (train_idx, valid_idx) in enumerate(folds, 1):

    train_idx = train_idx[
        known_mask.iloc[train_idx].to_numpy()
    ]
    valid_idx = valid_idx[
        known_mask.iloc[valid_idx].to_numpy()
    ]

    X_train = train_df.iloc[train_idx][features]
    X_valid = train_df.iloc[valid_idx][features]
    y_train = train_df.iloc[train_idx]["repair_cost"]

    model = CatBoostRegressor(
        loss_function="Quantile:alpha=0.90",
        iterations=300,
        depth=4,
        learning_rate=0.03,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )

    model.fit(X_train, y_train)

    cat_oof[valid_idx] = np.maximum(
        model.predict(X_valid), 0
    )

    print(f"CatBoost quantile: fold {fold_no}/5 complete")

actual = train_df.loc[known_mask, "repair_cost"].to_numpy()
predicted = cat_oof[known_mask.to_numpy()]

under = np.maximum(actual - predicted, 0)
over = np.maximum(predicted - actual, 0)
expensive = actual >= np.quantile(actual, 0.75)

cat_result = pd.DataFrame([{
    "model": "CatBoost_Q90",
    "MAE": mean_absolute_error(actual, predicted),
    "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
    "underestimation_rate": np.mean(predicted < actual),
    "asymmetric_loss": np.mean(0.20 * under + 0.02 * over),
    "expensive_quartile_MAE": mean_absolute_error(
        actual[expensive], predicted[expensive]
    ),
    "mean_prediction": predicted.mean(),
}])

display(cat_result.round(3))

CatBoost quantile: fold 1/5 complete
CatBoost quantile: fold 2/5 complete
CatBoost quantile: fold 3/5 complete
CatBoost quantile: fold 4/5 complete
CatBoost quantile: fold 5/5 complete


,model,MAE,RMSE,underestimation_rate,asymmetric_loss,expensive_quartile_MAE,mean_prediction
0,CatBoost_Q90,557.196,926.91,0.156,39.254,879.12,737.239


In [13]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from src.evaluation import tune_policy, calculate_cost

# Load existing production OOF failure probabilities.
final = pd.read_csv(
    ROOT / "results" / "final_budget_oof_predictions.csv"
).sort_values("id").reset_index(drop=True)

assert train_df["id"].reset_index(drop=True).equals(final["id"])

probabilities = final["oof_failure_probability"].to_numpy()
production_budget = final["oof_repair_budget"].to_numpy()

# Generate budget predictions for EVERY validation vehicle.
candidate_budgets = {
    "production": production_budget.copy(),
    "lgbm_q90": np.full(len(train_df), np.nan),
    "catboost_q90": np.full(len(train_df), np.nan),
}

for fold_no, (train_idx, valid_idx) in enumerate(folds, 1):

    fit_idx = train_idx[
        known_mask.iloc[train_idx].to_numpy()
    ]

    X_fit = train_df.iloc[fit_idx][features]
    y_fit = train_df.iloc[fit_idx]["repair_cost"]

    X_valid = train_df.iloc[valid_idx][features]

    # LightGBM quantile 0.90
    lgbm = LGBMRegressor(
        objective="quantile",
        alpha=0.90,
        n_estimators=200,
        learning_rate=0.03,
        num_leaves=7,
        min_child_samples=15,
        reg_lambda=5.0,
        verbosity=-1,
        random_state=42,
        n_jobs=-1,
    )
    lgbm.fit(X_fit, y_fit)

    candidate_budgets["lgbm_q90"][valid_idx] = np.maximum(
        lgbm.predict(X_valid), 0
    )

    # CatBoost quantile 0.90
    cat = CatBoostRegressor(
        loss_function="Quantile:alpha=0.90",
        iterations=300,
        depth=4,
        learning_rate=0.03,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )
    cat.fit(X_fit, y_fit)

    candidate_budgets["catboost_q90"][valid_idx] = np.maximum(
        cat.predict(X_valid), 0
    )

    print(f"All-vehicle OOF predictions: fold {fold_no}/5 complete")

# Cross-fitted policy evaluation.
policy_results = []

for name, budgets in candidate_budgets.items():

    assert np.isfinite(budgets).all()

    for heldout_fold, (train_idx, valid_idx) in enumerate(folds):

        policy = tune_policy(
            probabilities[train_idx],
            train_df.iloc[train_idx]["repair_cost"].to_numpy(),
            budgets[train_idx],
        )

        flags = (
            probabilities[valid_idx] >= policy["threshold"]
        ).astype(int)

        allocated = np.where(
            flags == 1,
            budgets[valid_idx] * policy["multiplier"],
            0.0,
        )

        cost = calculate_cost(
            train_df.iloc[valid_idx]["repair_cost"].to_numpy(),
            flags,
            allocated,
        )

        policy_results.append({
            "model": name,
            "fold": heldout_fold,
            "cost": cost,
            "inspections": flags.sum(),
            "threshold": policy["threshold"],
            "multiplier": policy["multiplier"],
        })

policy_df = pd.DataFrame(policy_results)

summary = (
    policy_df.groupby("model")
    .agg(
        total_cost=("cost", "sum"),
        mean_fold_cost=("cost", "mean"),
        total_inspections=("inspections", "sum"),
        mean_multiplier=("multiplier", "mean"),
    )
    .sort_values("total_cost")
)

summary["cost_per_vehicle"] = summary["total_cost"] / len(train_df)

display(summary.round(3))
display(
    policy_df.pivot(
        index="fold", columns="model", values="cost"
    ).round(2)
)

All-vehicle OOF predictions: fold 1/5 complete
All-vehicle OOF predictions: fold 2/5 complete
All-vehicle OOF predictions: fold 3/5 complete
All-vehicle OOF predictions: fold 4/5 complete
All-vehicle OOF predictions: fold 5/5 complete


,total_cost,mean_fold_cost,total_inspections,mean_multiplier,cost_per_vehicle
model,,,,,
catboost_q90,62929.702,12585.940,1229,0.50,1.124
production,63012.630,12602.526,1201,0.95,1.125
lgbm_q90,63335.839,12667.168,1201,0.50,1.131


model,catboost_q90,lgbm_q90,production
fold,,,
0,9652.34,9815.49,10262.59
1,10924.48,10826.43,10816.03
2,14326.86,14419.89,14299.76
3,16790.88,16680.86,16055.96
4,11235.14,11593.17,11578.28
